# Person Detection from NVIDIA Synthetic Data

Runtime -> Change runtime type -> **GPU (T4 x2)** before running.
Train on synthetic scenes, test generalization on held-out synthetic scenes,
and demo on real footage.


## 0. Setup


In [ ]:
!git clone https://github.com/yashas-y/RoboticsResearch.git
%cd RoboticsResearch
!pip install -q -r requirements.txt


In [ ]:
# (If you cloned earlier, pull the latest instead of cloning again.)
from huggingface_hub import notebook_login
notebook_login()


## 1. (optional) Explore the dataset layout


In [ ]:
!python src/explore_repo.py --grep MTMC_Tracking_2026/train --limit 20


## 2. Download small subsets (train + held-out test + real demo)
config.py is preset to skip the huge depth_maps and grab only a few cameras.


In [ ]:
!python src/download_data.py --which all


In [ ]:
# locate a label file to inspect next
!find data/raw/train -name 'ground_truth*'


## 3. Confirm the label schema
Paste one path from above into --path. If keys differ, fix parse_ground_truth()
in src/prepare_yolo.py.


In [ ]:
# !python src/inspect_annotations.py --path data/raw/train/<...>/ground_truth.json


## 4. Build the YOLO datasets


In [ ]:
!python src/prepare_yolo.py --split train
!python src/prepare_yolo.py --split test


## 5. Train on the synthetic scenes


In [ ]:
!python src/train.py --data data/yolo_train/data.yaml --name synth_all --epochs 50


## 6. Measure generalization (held-out synthetic scenes)


In [ ]:
!python src/evaluate.py --weights runs/detect/synth_all/weights/best.pt \
    --data data/yolo_test/data.yaml --tag synth_all --x 3


## 7. Real-world demo (annotated video on the REAL footage)


In [ ]:
!python src/predict_real.py --weights runs/detect/synth_all/weights/best.pt


## 8. Ablation: train on 1, 2, then all scenes -> data-scaling curve
See GUIDE.md Phase 9. Then plot:


In [ ]:
!python src/plot_results.py


In [ ]:
import pandas as pd
pd.read_csv('results/metrics.csv')
